# Projeto Final — Análise Exploratória de Dados (Olist)

## 1. Apresentação da Base e Perguntas de Negócio
### Origem dos Dados
Os dados foram extraídos do dataset público do **Olist** no Kaggle, cobrindo o e-commerce brasileiro.

### Perguntas de Negócio
1. **Prazos e Frete:** Qual a relação entre o custo do frete e o tempo real de entrega entre os diferentes estados?
2. **Meios de Pagamento e Ticket:** Como o valor médio do pedido varia em função do tipo de pagamento e do parcelamento?
3. **Outliers de Frete:** Quais estados apresentam os maiores outliers de valor de frete em relação ao preço das mercadorias?
4. **Recorrência:** Clientes recorrentes possuem comportamento de compra diferente dos clientes pontuais?

In [14]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Leitura das bases de dados (existentes + novas)
df_customers = pd.read_csv('../dados/olist_customers_dataset.csv')
df_orders = pd.read_csv('../dados/olist_orders_dataset.csv')
df_items = pd.read_csv('../dados/olist_order_items_dataset.csv')
df_payments = pd.read_csv('../dados/olist_order_payments_dataset.csv')

# Bases adicionais
df_reviews = pd.read_csv('../dados/olist_order_reviews_dataset.csv')
df_products = pd.read_csv('../dados/olist_products_dataset.csv')
df_categories = pd.read_csv('../dados/product_category_name_translation.csv')

print("Todas as 7 bases foram carregadas com sucesso!")

Todas as 7 bases foram carregadas com sucesso!


Antes de realizar os cruzamentos entre as tabelas, vamos verificar a estrutura e a granularidade de cada base. Isso é importante porque uma mesma informação pode aparecer várias vezes em algumas tabelas. Por exemplo, um pedido pode possuir vários itens e também vários pagamentos. Portanto, em vez de criar um único DataFrame com todas as tabelas, construiremos DataFrames específicos de acordo com cada pergunta de negócio, evitando a duplicação de registros e a distorção das métricas.

In [15]:
# Tradução das categorias dos produtos
df_products = df_products.merge(
    df_categories,
    on='product_category_name',
    how='left'
)

print("Categorias dos produtos traduzidas.")

Categorias dos produtos traduzidas.


## Verificação da estrutura das bases

Primeiro, vamos verificar a quantidade de linhas e colunas de cada tabela. Essa informação ajuda a identificar o tamanho das bases e, principalmente, perceber que elas possuem diferentes níveis de granularidade.

In [16]:
print("orders:", df_orders.shape)
print("customers:", df_customers.shape)
print("items:", df_items.shape)
print("payments:", df_payments.shape)
print("products:", df_products.shape)
print("reviews:", df_reviews.shape)
print("categories:", df_categories.shape)

orders: (99441, 8)
customers: (99441, 5)
items: (112650, 7)
payments: (103886, 5)
products: (32951, 10)
reviews: (99224, 7)
categories: (71, 2)


# Verificação da granularidade

Agora vamos verificar se order_id é único em cada tabela. Em orders, esperamos um registro por pedido. Já em items e payments, um mesmo pedido pode aparecer várias vezes, pois um pedido pode conter vários itens e pode possuir mais de um pagamento.

In [13]:
print("order_id único em orders:",
      df_orders['order_id'].is_unique)

print("order_id único em items:",
      df_items['order_id'].is_unique)

print("order_id único em payments:",
      df_payments['order_id'].is_unique)

order_id único em orders: True
order_id único em items: False
order_id único em payments: False


Interpretação: A tabela orders possui um registro único para cada pedido. Já items e payments possuem múltiplos registros para um mesmo order_id. Por isso, juntar essas tabelas diretamente poderia multiplicar registros quando um pedido possui vários itens e vários pagamentos. Para evitar esse problema, as tabelas serão combinadas de acordo com a necessidade de cada análise.

## 2. Construção dos DataFrames para análise

Como as tabelas possuem diferentes níveis de granularidade, não utilizaremos um único DataFrame com todas as informações.

Os cruzamentos serão realizados de acordo com cada pergunta de negócio, preservando a granularidade adequada para cada análise.

### 2.1 Pedidos e clientes

Primeiro, vamos relacionar os pedidos aos seus respectivos clientes.

A tabela `orders` possui um registro por pedido e `customers` possui as informações do cliente associado ao pedido. Portanto, esperamos uma relação de um para um entre essas duas tabelas por `customer_id`.

In [21]:
df_orders_customers = df_orders.merge(
    df_customers,
    on='customer_id',
    how='left',
    validate='one_to_one'
)

print("Orders:", df_orders.shape)
print("Orders + Customers:", df_orders_customers.shape)

Orders: (99441, 8)
Orders + Customers: (99441, 12)


**Interpretação:** O primeiro cruzamento entre `orders` e `customers` manteve as 99.441 linhas da tabela de pedidos. Isso confirma que o relacionamento utilizado preservou a granularidade de um registro por pedido.

### 2.2 Pedidos, clientes e itens

Agora adicionaremos os itens dos pedidos. Diferentemente da relação anterior, um pedido pode possuir vários itens. Portanto, esperamos uma relação de um para muitos (`one-to-many`).

Após esse cruzamento, a granularidade do DataFrame passa de **um registro por pedido** para **um registro por item do pedido**.

In [18]:
df_entrega = df_orders_customers.merge(
    df_items,
    on='order_id',
    how='left',
    validate='one_to_many'
)

print("Orders + Customers:", df_orders_customers.shape)
print("Orders + Customers + Items:", df_entrega.shape)

Orders + Customers: (99441, 12)
Orders + Customers + Items: (113425, 18)


**Interpretação:** Após adicionar a tabela `items`, o número de registros aumentou de 99.441 para 113.425. Isso ocorre porque um mesmo pedido pode conter vários itens. Portanto, o DataFrame `df_entrega` passa a ter como granularidade um registro por item do pedido, e não mais um registro por pedido.

In [22]:
df_entrega[['order_id', 'product_id', 'price', 'freight_value']].head(10)

,order_id,product_id,price,freight_value
0,e481f51cbdc54678b7cc49136f2d6af7,87285b34884572647811a353c7ac498a,29.99,8.72
1,53cdb2fc8bc7dce0b6741e2150273451,595fac2a385ac33a80bd5114aec74eb8,118.70,22.76
2,47770eb9100c2d0c44946d9cf07ec65d,aa4383b373c6aca5d8797843e5594415,159.90,19.22
3,949d5b44dbf5de918fe9c16f97b45f8a,d0b61bfb1de832b15ba9d266ca96e5b0,45.00,27.20
4,ad21c59c0840e6cb83a9ceb5573f8159,65266b2da20d04dbe00c5c2d3bb7859e,19.90,8.72
5,a4591c265e18cb1dcee52889e2d8acc3,060cb19345d90064d1015407193c233d,147.90,27.36
6,136cce7faa42fdb2cefd53fdc79a6098,a1804276d9941ac0733cfd409f5206eb,49.90,16.05
7,6514b8ad8028c9f2cc2374ded245783f,4520766ec412348b8d4caa5e8a18c464,59.99,15.17
8,76c6e866289321a7c93b82b54852dc33,ac1789e492dcd698c5c10b97a671243a,19.90,16.05
9,e69bfb5eb88e0ed6a785585b27e16dbf,9a78fb9862b10749a117f7fc3c31f051,149.99,19.77


### Verificação de um pedido com múltiplos itens

Para visualizar na prática a mudança de granularidade, vamos identificar um pedido que possui vários itens e observar como ele aparece no DataFrame após o cruzamento.

In [23]:
pedido_exemplo = (
    df_entrega.groupby('order_id')
    .size()
    .sort_values(ascending=False)
    .index[0]
)

print("Pedido escolhido:", pedido_exemplo)

df_entrega[
    df_entrega['order_id'] == pedido_exemplo
][['order_id', 'product_id', 'price', 'freight_value']]

Pedido escolhido: 8272b63d03f5f79c56e9e4120aec44ef


,order_id,product_id,price,freight_value
101222,8272b63d03f5f79c56e9e4120aec44ef,270516a3f41dc035aa87d220228f844c,1.2,7.89
101223,8272b63d03f5f79c56e9e4120aec44ef,05b515fdc76e888aada3c6d66c201dff,1.2,7.89
101224,8272b63d03f5f79c56e9e4120aec44ef,05b515fdc76e888aada3c6d66c201dff,1.2,7.89
101225,8272b63d03f5f79c56e9e4120aec44ef,05b515fdc76e888aada3c6d66c201dff,1.2,7.89
101226,8272b63d03f5f79c56e9e4120aec44ef,05b515fdc76e888aada3c6d66c201dff,1.2,7.89
101227,8272b63d03f5f79c56e9e4120aec44ef,05b515fdc76e888aada3c6d66c201dff,1.2,7.89
101228,8272b63d03f5f79c56e9e4120aec44ef,05b515fdc76e888aada3c6d66c201dff,1.2,7.89
101229,8272b63d03f5f79c56e9e4120aec44ef,05b515fdc76e888aada3c6d66c201dff,1.2,7.89
101230,8272b63d03f5f79c56e9e4120aec44ef,05b515fdc76e888aada3c6d66c201dff,1.2,7.89
101231,8272b63d03f5f79c56e9e4120aec44ef,05b515fdc76e888aada3c6d66c201dff,1.2,7.89


### 2.3 Itens e produtos

Para complementar as informações dos itens, vamos relacioná-los aos dados dos produtos por meio de `product_id`.

A tabela `order_items` pode possuir vários registros para o mesmo produto, pois um produto pode ser vendido em diferentes pedidos. Já a tabela `products` possui um registro por produto.

Portanto, esperamos uma relação de muitos para um (`many-to-one`).

In [24]:
df_itens_produtos = df_items.merge(
    df_products,
    on='product_id',
    how='left',
    validate='many_to_one'
)

print("Items:", df_items.shape)
print("Items + Products:", df_itens_produtos.shape)

Items: (112650, 7)
Items + Products: (112650, 16)


O cruzamento manteve a quantidade de registros porque cada item está associado a um único produto. A relação `many-to-one` foi validada com sucesso.

## 3. Diagnóstico da Qualidade dos Dados

Antes de realizar as análises, vamos verificar a qualidade e a estrutura dos dados utilizados.

Como cada pergunta de negócio utiliza DataFrames com diferentes granularidades, o diagnóstico será realizado considerando a base correspondente a cada análise.

Nesta primeira etapa, vamos analisar o `df_entrega`, utilizado nas análises relacionadas a prazo, frete e estado.

### 3.1 Dimensões, tipos e uso de memória

In [25]:
df_entrega.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 113425 entries, 0 to 113424
Data columns (total 18 columns):
 #   Column                         Non-Null Count   Dtype  
---  ------                         --------------   -----  
 0   order_id                       113425 non-null  object 
 1   customer_id                    113425 non-null  object 
 2   order_status                   113425 non-null  object 
 3   order_purchase_timestamp       113425 non-null  object 
 4   order_approved_at              113264 non-null  object 
 5   order_delivered_carrier_date   111457 non-null  object 
 6   order_delivered_customer_date  110196 non-null  object 
 7   order_estimated_delivery_date  113425 non-null  object 
 8   customer_unique_id             113425 non-null  object 
 9   customer_zip_code_prefix       113425 non-null  int64  
 10  customer_city                  113425 non-null  object 
 11  customer_state                 113425 non-null  object 
 12  order_item_id                 

**Interpretação:** O DataFrame `df_entrega` possui 113.425 registros e 18 colunas, com aproximadamente 15,6 MB de uso de memória. A maior parte das colunas está no formato `object`, incluindo as datas, que posteriormente poderão ser convertidas para o formato `datetime` para permitir cálculos relacionados a prazos de entrega.

Também foram identificados valores ausentes em algumas colunas, principalmente nas informações relacionadas à aprovação e às datas de entrega. Esses valores serão investigados na próxima etapa do diagnóstico antes de definir qualquer estratégia de tratamento.

### 3.2 Valores ausentes

In [26]:
missing = pd.DataFrame({
    'quantidade': df_entrega.isna().sum(),
    'percentual': (df_entrega.isna().mean() * 100).round(2)
})

missing = missing[missing['quantidade'] > 0].sort_values(
    'percentual',
    ascending=False
)

missing

,quantidade,percentual
order_delivered_customer_date,3229,2.85
order_delivered_carrier_date,1968,1.74
order_item_id,775,0.68
product_id,775,0.68
seller_id,775,0.68
shipping_limit_date,775,0.68
price,775,0.68
freight_value,775,0.68
order_approved_at,161,0.14


### Investigação dos valores ausentes em `order_items`

As colunas relacionadas aos itens apresentam exatamente a mesma quantidade de valores ausentes. Vamos verificar se esses registros correspondem a pedidos que não possuem informações na tabela `order_items`.

Essa verificação é importante para diferenciar valores ausentes originados nos dados de origem daqueles que surgiram como consequência do cruzamento entre as tabelas.

In [27]:
colunas_items = [
    'order_item_id',
    'product_id',
    'seller_id',
    'shipping_limit_date',
    'price',
    'freight_value'
]

df_entrega[df_entrega['order_item_id'].isna()][colunas_items].head()

,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
306,NaN,NaN,NaN,NaN,NaN,NaN
671,NaN,NaN,NaN,NaN,NaN,NaN
791,NaN,NaN,NaN,NaN,NaN,NaN
850,NaN,NaN,NaN,NaN,NaN,NaN
1294,NaN,NaN,NaN,NaN,NaN,NaN


### Investigação dos valores ausentes nas datas do pedido

As datas relacionadas ao processo de entrega apresentam valores ausentes em diferentes proporções. Antes de definir qualquer tratamento, vamos verificar a situação dos pedidos e identificar se os valores ausentes estão relacionados ao status do pedido.

In [28]:
df_entrega['order_status'].value_counts(dropna=False)

order_status
delivered      110197
shipped          1186
canceled          706
unavailable       610
invoiced          361
processing        357
created             5
approved            3
Name: count, dtype: int64

### Relação entre status do pedido e datas ausentes

Como `df_entrega` possui granularidade de item, a análise das datas e do status será realizada utilizando `df_orders_customers`, que possui uma linha por pedido.

Dessa forma, cada pedido será contabilizado uma única vez, evitando que pedidos com vários itens tenham peso maior no diagnóstico.

In [32]:
status_datas = df_orders_customers.groupby('order_status').agg(
    total_pedidos=('order_id', 'count'),
    sem_data_transportadora=('order_delivered_carrier_date', lambda x: x.isna().sum()),
    sem_data_entrega=('order_delivered_customer_date', lambda x: x.isna().sum())
)

status_datas

,total_pedidos,sem_data_transportadora,sem_data_entrega
order_status,,,
approved,2,2,2
canceled,625,550,619
created,5,5,5
delivered,96478,2,8
invoiced,314,314,314
processing,301,301,301
shipped,1107,0,1107
unavailable,609,609,609


### Investigação dos pedidos entregues sem data de entrega

Embora a maioria dos pedidos com status `delivered` possua as datas de entrega preenchidas, foram identificados 8 pedidos sem `order_delivered_customer_date`.

Vamos verificar esses registros antes de definir o tratamento, pois eles não poderão ser utilizados no cálculo do prazo real de entrega enquanto a data estiver ausente.

In [33]:
df_orders_customers[
    (df_orders_customers['order_status'] == 'delivered') &
    (df_orders_customers['order_delivered_customer_date'].isna())
][[
    'order_id',
    'order_status',
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]]

,order_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
3002,2d1e2d5bf4dc7227b3bfebb81328c15f,delivered,2017-11-28 17:44:07,2017-11-28 17:56:40,2017-11-30 18:12:23,NaN,2017-12-18 00:00:00
20618,f5dd62b788049ad9fc0526e3ad11a097,delivered,2018-06-20 06:58:43,2018-06-20 07:19:05,2018-06-25 08:05:00,NaN,2018-07-16 00:00:00
43834,2ebdfc4f15f23b91474edf87475f108e,delivered,2018-07-01 17:05:11,2018-07-01 17:15:12,2018-07-03 13:57:00,NaN,2018-07-30 00:00:00
79263,e69f75a717d64fc5ecdfae42b2e8e086,delivered,2018-07-01 22:05:55,2018-07-01 22:15:14,2018-07-03 13:57:00,NaN,2018-07-30 00:00:00
82868,0d3268bad9b086af767785e3f0fc0133,delivered,2018-07-01 21:14:02,2018-07-01 21:29:54,2018-07-03 09:28:00,NaN,2018-07-24 00:00:00
92643,2d858f451373b04fb5c984a1cc2defaf,delivered,2017-05-25 23:22:43,2017-05-25 23:30:16,NaN,NaN,2017-06-23 00:00:00
97647,ab7c89dc1bf4a1ead9d6ec1ec8968a84,delivered,2018-06-08 12:09:39,2018-06-08 12:36:39,2018-06-12 14:10:00,NaN,2018-06-26 00:00:00
98038,20edc82cf5400ce95e1afacc25798b31,delivered,2018-06-27 16:09:12,2018-06-27 16:29:30,2018-07-03 19:26:00,NaN,2018-07-19 00:00:00


**Interpretação:** Foram identificados 8 pedidos com status `delivered` que não possuem a data efetiva de entrega ao cliente. Como essa informação não pode ser determinada de forma confiável a partir dos dados disponíveis, esses valores não serão preenchidos artificialmente.

Para a análise de prazo real de entrega, serão considerados posteriormente apenas os pedidos que possuam as datas necessárias para o cálculo. Dessa forma, preservamos os dados originais e evitamos introduzir informações estimadas sem justificativa.

### 3.3 Registros duplicados

Como `df_orders_customers` possui uma linha por pedido, vamos verificar se existem registros completamente duplicados nessa base.

Essa verificação é importante para identificar possíveis repetições dos mesmos registros antes das etapas de transformação e análise.

In [35]:
duplicados_orders = df_orders_customers.duplicated().sum()

print("Registros duplicados:", duplicados_orders)

Registros duplicados: 0


**Interpretação:** Não foram identificados registros completamente duplicados em `df_orders_customers`. Portanto, não será necessário realizar uma remoção de duplicidades nessa base.

### 3.4 Verificação de categorias inconsistentes

Vamos verificar as categorias existentes em `order_status` para identificar possíveis diferenças de grafia, espaços ou outras inconsistências que possam representar a mesma categoria.

In [36]:
df_orders_customers['order_status'].value_counts(dropna=False)

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

**Interpretação:** As categorias de `order_status` apresentam nomenclatura padronizada, sem diferenças aparentes de grafia, capitalização ou espaços que indiquem categorias duplicadas. Portanto, não é necessário realizar tratamento de padronização nessa variável.

### Verificação das categorias de estado

Como o estado do cliente será utilizado nas análises de frete e prazo, vamos verificar as categorias presentes em `customer_state` e identificar possíveis valores inconsistentes.

In [37]:
df_orders_customers['customer_state'].value_counts(dropna=False)

customer_state
SP    41746
RJ    12852
MG    11635
RS     5466
PR     5045
SC     3637
BA     3380
DF     2140
ES     2033
GO     2020
PE     1652
CE     1336
PA      975
MT      907
MA      747
MS      715
PB      536
PI      495
RN      485
AL      413
SE      350
TO      280
RO      253
AM      148
AC       81
AP       68
RR       46
Name: count, dtype: int64

### Verificação de espaços e caps

Além da inspeção das categorias, vamos verificar se existem valores que seriam alterados ao remover espaços ou padronizar as letras para maiúsculas.

In [38]:
estados_inconsistentes = df_orders_customers[
    df_orders_customers['customer_state'] !=
    df_orders_customers['customer_state'].str.strip().str.upper()
][['customer_state']]

estados_inconsistentes.drop_duplicates()

,customer_state


### 3.5 Verificação de valores inválidos

Nesta etapa, vamos verificar se existem valores numéricos que não são compatíveis com o significado das variáveis utilizadas na análise.

Como `price` representa o preço do item e `freight_value` representa o valor do frete, valores negativos seriam inconsistentes com essas definições.

In [39]:
valores_invalidos = {
    'price_negativo': (df_entrega['price'] < 0).sum(),
    'freight_negativo': (df_entrega['freight_value'] < 0).sum()
}

pd.Series(valores_invalidos)

price_negativo      0
freight_negativo    0
dtype: int64

### 2.2 Identificação de Outliers

#### Método IQR

In [ ]:
# Seleciona apenas colunas numéricas
colunas_numericas = df.select_dtypes(include="number").columns

# Opcional: remova identificadores ou códigos que não devem ser analisados
# colunas_numericas = colunas_numericas.drop(["order_item_id"])

outliers = pd.DataFrame(False, index=df.index, columns=colunas_numericas)
limites = {}

for coluna in colunas_numericas:
    q1 = df[coluna].quantile(0.25)
    q3 = df[coluna].quantile(0.75)
    iqr = q3 - q1

    limite_inferior = q1 - 1.5 * iqr
    limite_superior = q3 + 1.5 * iqr

    limites[coluna] = (limite_inferior, limite_superior)

    outliers[coluna] = (
        (df[coluna] < limite_inferior) |
        (df[coluna] > limite_superior)
    )

# Quantidade de outliers por coluna
quantidade_outliers = outliers.sum().sort_values(ascending=False)
print(quantidade_outliers[quantidade_outliers > 0])

In [ ]:
df_outliers = df[outliers.any(axis=1)]

print(df_outliers.shape)
display(df_outliers.head())

In [ ]:
df[colunas_numericas].boxplot(figsize=(14, 6), rot=45)
plt.title("Boxplots das variáveis numéricas")
plt.show()

#### Método Z-Score

In [ ]:
# Seleciona apenas colunas numéricas
colunas_numericas = df.select_dtypes(include="number").columns

# Opcional: remova identificadores ou códigos que não devem ser analisados
# colunas_numericas = colunas_numericas.drop(["order_item_id"])

# Calcula o z-score de cada valor em relação à sua coluna
media = df[colunas_numericas].mean()
desvio_padrao = df[colunas_numericas].std()
z_scores = (df[colunas_numericas] - media) / desvio_padrao

# Considera outlier todo valor com z-score absoluto maior que 3
limite_z = 3
outliers_z = z_scores.abs() > limite_z

# Quantidade de outliers por coluna
quantidade_outliers_z = outliers_z.sum().sort_values(ascending=False)
print(quantidade_outliers_z[quantidade_outliers_z > 0])

# Linhas que possuem pelo menos um outlier
df_outliers_z = df[outliers_z.any(axis=1)]
print(f"\\nLinhas com pelo menos um outlier: {df_outliers_z.shape[0]}")
display(df_outliers_z.head())

Próximo passo: Analisar a distribuição de cada variável quantitativa para determinar o tipo (normal ou assimétrica)

In [ ]:
# 2.3 Análise de Distribuição das Variáveis Quantitativas

# Seleção das colunas numéricas (excluindo IDs sequenciais/códigos se houver)
colunas_numericas = df.select_dtypes(include=["number"]).columns
if 'order_item_id' in colunas_numericas:
    colunas_numericas = colunas_numericas.drop(['order_item_id'])

# Cálculo das métricas de Assimetria (Skewness) e Curtose (Kurtosis)
dados_distribuicao = []

for col in colunas_numericas:
    skew = df[col].skew()
    kurt = df[col].kurt()
    
    # Classificação baseada no Skewness
    if abs(skew) < 0.5:
        tipo_distribuicao = "Aproximadamente Normal"
    elif skew >= 0.5:
        tipo_distribuicao = "Assimétrica à Direita (Cauda Longa)"
    else:
        tipo_distribuicao = "Assimétrica à Esquerda"
        
    dados_distribuicao.append({
        'Variável': col,
        'Média': df[col].mean(),
        'Mediana': df[col].median(),
        'Desvio Padrão': df[col].std(),
        'Skewness (Assimetria)': round(skew, 2),
        'Kurtosis': round(kurt, 2),
        'Classificação': tipo_distribuicao
    })

df_distribuicao = pd.DataFrame(dados_distribuicao)
display(df_distribuicao)

In [ ]:
# Plot dos histogramas com estimativa de densidade de kernel (KDE)
plt.figure(figsize=(16, 10))
num_cols = len(colunas_numericas)
cols_per_row = 3
rows = (num_cols + cols_per_row - 1) // cols_per_row

for i, col in enumerate(colunas_numericas, 1):
    plt.subplot(rows, cols_per_row, i)
    sns.histplot(df[col], kde=True, bins=30, color='skyblue')
    plt.title(f'Distribuição de {col}')
    plt.xlabel(col)
    plt.ylabel('Frequência')

plt.tight_layout()
plt.show()

## 3 Limpeza e transformação

In [ ]:
# 1. Lista de todas as colunas com tipos de dados e contagem de não-nulos
print("--- INFORMAÇÕES DO DATASET CONSOLIDADO ---")
df.info(verbose=True, show_counts=True)

In [ ]:
# 2. Amostra de 5 linhas com todas as colunas visíveis
pd.set_option('display.max_columns', None)
df.head(5)

In [ ]:
# 1. Seleção das colunas essenciais para as Perguntas de Negócio
colunas_essenciais = [
    'order_id',
    'customer_unique_id',
    'customer_state',
    'order_purchase_timestamp',
    'order_delivered_customer_date',
    'order_estimated_delivery_date',
    'price',
    'freight_value',
    'payment_type',
    'payment_installments',
    'payment_value',
    'product_category_name',
    'product_category_name_english',
    'review_score'
]

# Filtra o DataFrame principal para manter somente as colunas selecionadas
df = df[colunas_essenciais].copy()

# 2. Conversão das colunas de data para datetime
colunas_datas = [
    'order_purchase_timestamp',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]

for col in colunas_datas:
    df[col] = pd.to_datetime(df[col])

# 3. Criação de Métricas de Entrega
# Tempo real de entrega em dias
df['tempo_entrega_dias'] = (df['order_delivered_customer_date'] - df['order_purchase_timestamp']).dt.total_seconds() / (24 * 3600)

# Diferença do prazo estimado em dias (positivo = atraso / negativo = adiantado)
df['dias_diferenca_estimada'] = (df['order_delivered_customer_date'] - df['order_estimated_delivery_date']).dt.total_seconds() / (24 * 3600)

# Indicador de entrega com atraso
df['entregue_com_atraso'] = df['dias_diferenca_estimada'] > 0

# 4. Preenchimento simples de nulos em categorias
df['product_category_name'] = df['product_category_name'].fillna('outro')
df['product_category_name_english'] = df['product_category_name_english'].fillna('other')

print("--- DATASET REDUZIDO E LIMPO ---")
print(f"Estrutura final: {df.shape[0]} linhas x {df.shape[1]} colunas")
df.head(3)

## 4 Análise exploratória

## 5 Conclusões